# Data cleaning

Turns the raw per-run output of AmberTorchPB (called from AMBER PBSA; one row per
molecule × grid spacing × tolerance × device × solver) into the cleaned table the
analysis notebooks read.

The runs follow the benchmark design of the paper (Section 3.1.1): the nucleic acid
and protein splits of the AMBER PBSA benchmark suite, grid spacings of 0.35 / 0.55 /
0.75 / 0.95 Å, and convergence tolerances of 1e-4 (default), 1e-5 and 1e-6.

The same steps are packaged as `torchpb_datatool.clean_raw`; a cell near the end
checks that the two agree.

In [ ]:
import pandas as pd

from torchpb_datatool import clean_raw
from torchpb_datatool.schema import CLEANED_COLUMNS, RAW_TO_CLEAN, SOLVER_INDEX_TO_NAME, SYSTEM_TO_SPLIT, solver_label

raw = pd.read_csv("data/sample_raw.csv")  # default: sample raw data from scripts/generate_sample_data.py; replace with your own raw experiment CSV
df = raw.copy()

df.sample(5)

## Molecule and solver from the file path

Paths look like `/…/solver_<solver_opt>/<molecule>.p22_g<spacing × 100>.out`.

In [ ]:
df["Mol Name"] = df["file"].str.extract(r"/([^/]+)_g\d{2,4}", expand=False)
print(df[["file", "Mol Name"]].head())

In [ ]:
df["Solver Name"] = df["file"].str.extract(r"/solver_(\d)/", expand=False)
df.head()

## Rename to the cleaned schema

`grid` is the grid spacing in Å, `accept` the convergence tolerance on
‖Ax − b‖₁ / ‖b‖₁, `Total_time` the solver time in seconds, and `itn` (optional) the
iteration count.

In [ ]:
df = df.rename(columns=RAW_TO_CLEAN)
df.head()

## Failed runs

A run without an energy did not converge (GMRES and SOR occasionally fail,
Section 3.3.1). `Succeeded` is derived from the energy rather than from `Run_Done`.

In [ ]:
df[df["Elapsed Time"].isna()]

In [ ]:
df[df["EPB Value"].isna()]

In [ ]:
df["Succeeded"] = (~df["EPB Value"].isna()).astype(int)

## Split and solver labels

`solver_opt` indices follow the AmberTorchPB wrapper: 0 CG, 1 BiCG (BiCGSTAB),
2 GMRES, 3 SGD, 4 Adam, 5 neural network, 6 SOR, 7 red-black SOR. The paper
benchmarks CG, BiCG, GMRES, SOR and RB-SOR.

In [ ]:
df["Split"] = df["Split"].replace(SYSTEM_TO_SPLIT)
df["Solver Name"] = df["Solver Name"].replace(SOLVER_INDEX_TO_NAME)
df[["Split", "Solver Name"]].value_counts().sort_index()

## Final columns

In [ ]:
columns = CLEANED_COLUMNS + (["Iterations"] if "Iterations" in df.columns else [])
df = df[columns].copy()
df["Mol Name"] = df["Mol Name"].str.replace(".p22", "", regex=False)
df["Elapsed Time"] = df["Elapsed Time"].round(2)
df["EPB Value"] = df["EPB Value"].round(4)
df["Sor Omega"] = pd.NA
df["Other Comments"] = pd.NA
df.head()

In [ ]:
# the packaged pipeline gives the same table
pd.testing.assert_frame_equal(df, clean_raw(raw))

## Export

One file with every run, plus one per solver (`cleaned_cg.csv`, `cleaned_bicg.csv`,
`cleaned_gmres.csv`, `cleaned_sor.csv`, `cleaned_rb-sor.csv`, …).

In [ ]:
# default output paths; change as needed
df.to_csv("data/cleaned_all.csv", index=False, encoding="utf-8-sig")
for name, part in df.groupby("Solver Name"):
    path = f"data/cleaned_{solver_label(name).lower()}.csv"
    part.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"{path}: {len(part)} rows")